<div style="padding:48px 42px;border-radius:18px;background:linear-gradient(120deg,#071b2c,#0b394a 68%,#087e78);color:white;border:1px solid #35cdbd">
<p style="letter-spacing:.18em;color:#56e4d1;font-weight:700">RELATÓRIO DE INDICADORES · BRASIL</p>
<h1 style="font-size:52px;line-height:1.0;margin:20px 0;color:white">RAIO-X DA<br>JUVENTUDE</h1>
<h2 style="color:#c8e8e2;font-weight:500">Educação, Tecnologia e Oportunidades para o Futuro</h2>
<p style="max-width:760px;color:#c3d6d6">Análise descritiva das condições de vida, educação, inclusão digital e oportunidades de estudo e trabalho de jovens brasileiros, com indicadores publicados pelo IBGE e pelo INEP.</p>
<p style="color:#8fb4b3">PNAD Contínua · Censo Escolar · Brasil · 2016–2025</p></div>

**Responsabilidade com os dados.** Os valores organizados neste notebook foram fornecidos como transcrições de publicações oficiais. Neste ambiente, não foi possível acessar a API do IBGE nem conferir visualmente as tabelas originais. As fontes oficiais estão documentadas ao longo do projeto e a conferência final dos quadros e notas metodológicas é necessária antes da apresentação. A notícia da CONFEST/CONFEGE é contexto institucional, não fonte estatística.

## 1. Objetivo e metodologia
O projeto descreve dimensões importantes das oportunidades juvenis: utilização da internet, combinação de estudo e ocupação, média de anos de estudo e infraestrutura de conexão nas escolas. A unidade de análise muda entre indicadores; por isso, resultados de pessoas, estudantes e escolas são apresentados em painéis separados.

A abordagem é **descritiva**. Não foram processados microdados, calculadas estimativas próprias ou identificadas relações causais. Os números são transcritos dos resumos fornecidos e precisam ser conferidos nos documentos originais. A internet geral abrange pessoas de 10 anos ou mais; educação e trabalho abrangem jovens de 15 a 29 anos; anos de estudo, jovens de 18 a 29 anos; banda larga, escolas de educação infantil.

### Contextualização institucional
A notícia *“IBGE detalha em coletivas públicas a programação da CONFEST/CONFEGE sobre juventude e tecnologia”*, indicada com data de 29/10/2026, situa institucionalmente o debate sobre juventude e tecnologia. Ela não é tratada como levantamento estatístico. O link específico da notícia não foi validado; o portal da Agência de Notícias do IBGE está listado nas referências para localização.

In [ ]:
# 2. Importação das bibliotecas
from pathlib import Path
import json
import shutil
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid', context='notebook')
PALETA = {'marinho': '#0B2B40', 'turquesa': '#20B8A7', 'menta': '#83D9C8', 'lima': '#B3DF76', 'coral': '#F19A83'}
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 220, 'axes.titleweight': 'bold', 'axes.labelcolor': '#426071', 'text.color': '#112B3D', 'axes.edgecolor': '#DCE5E2', 'font.family': 'DejaVu Sans'})
print('Bibliotecas carregadas; paleta visual definida.')

In [ ]:
# 3. Configuração das pastas
PASTA_PROJETO = Path('/content/raio_x_juventude') if Path('/content').exists() else Path.cwd() / 'raio_x_juventude'
PASTA_DADOS = PASTA_PROJETO / 'data'
PASTA_GRAFICOS = PASTA_PROJETO / 'assets' / 'graficos'
PASTA_DADOS.mkdir(parents=True, exist_ok=True)
PASTA_GRAFICOS.mkdir(parents=True, exist_ok=True)
print(f'Arquivos serão preparados em: {PASTA_PROJETO.resolve()}')

## 2. Coleta e documentação dos dados oficiais
Não há coleta automática nesta versão: as estatísticas foram transcritas do material fornecido e organizadas abaixo. A tentativa de consulta à API SIDRA/IBGE não foi concluída por indisponibilidade de conexão deste ambiente; não substituí a consulta por números estimados. Use os links para conferir os quadros publicados, as populações de referência e eventuais revisões.

| Indicador | Fonte e link | Referência / população | Unidade e definição |
|---|---|---|---|
| Uso da internet | [IBGE, PNAD Contínua TIC](https://www.ibge.gov.br/estatisticas/sociais/educacao/17270-pnad-continua.html) | 2016, 2019, 2023, 2024, 2025; pessoas de 10 anos ou mais | Percentual que utilizou internet |
| Estudo e ocupação | [IBGE, PNAD Contínua Educação](https://www.ibge.gov.br/estatisticas/sociais/educacao/17270-pnad-continua.html) | 2024; jovens de 15 a 29 anos | Distribuição percentual por condição de estudo/qualificação e ocupação |
| Anos de estudo | [IBGE, PNAD Contínua Educação](https://www.ibge.gov.br/estatisticas/sociais/educacao/17270-pnad-continua.html) | 2016 e 2024; jovens de 18 a 29 anos por grupo | Média de anos de estudo |
| Banda larga escolar | [INEP, resultados do Censo Escolar](https://www.gov.br/inep/pt-br/areas-de-atuacao/pesquisas-estatisticas-e-indicadores/censo-escolar/resultados) | 2024; escolas de educação infantil | Percentual de escolas com internet banda larga |

**Limitação de rastreabilidade:** os portais institucionais são links oficiais para localizar a publicação, mas não confirmam isoladamente os valores transcritos. A referência indicada para o gráfico 52 do Resumo Técnico do Censo Escolar 2024 deve ser conferida no PDF original.

In [ ]:
# 4. Valores transcritos para análise; não são uma extração automática de microdados.
internet = pd.DataFrame({
    'ano': [2016, 2019, 2023, 2024, 2025],
    'uso_internet_percentual': [66.0, 79.4, 88.0, 89.1, 90.5],
    'populacao': 'Pessoas de 10 anos ou mais',
    'fonte': 'IBGE - PNAD Contínua TIC'
})

internet_recortes = pd.DataFrame({
    'grupo': ['Área urbana', 'Área rural', 'Estudantes da rede pública', 'Estudantes da rede privada'],
    'percentual': [91.5, 83.0, 89.9, 97.2],
    'ano': 2025,
    'fonte': 'IBGE - PNAD Contínua TIC'
})

estudo_trabalho = pd.DataFrame({
    'situacao': [
        'Ocupados e estudavam ou se qualificavam',
        'Não ocupados e não estudavam nem se qualificavam',
        'Não ocupados e estudavam ou se qualificavam',
        'Ocupados e não estudavam nem se qualificavam'
    ],
    'percentual': [16.4, 18.5, 25.3, 39.4],
    'ano': 2024,
    'populacao': 'Pessoas de 15 a 29 anos (estimativa informada: 48,0 milhões)',
    'fonte': 'IBGE - PNAD Contínua Educação'
})

escolaridade = pd.DataFrame({
    'grupo': ['Brasil', 'Branca', 'Preta ou parda', 'Área urbana', 'Área rural', 'Menor quartil de renda', 'Maior quartil de renda'],
    2016: [11.1, 11.9, 10.5, 11.4, 9.4, 9.3, 13.3],
    2024: [11.9, 12.5, 11.5, 12.1, 10.5, 10.6, 13.5],
    'populacao': 'Pessoas de 18 a 29 anos',
    'fonte': 'IBGE - PNAD Contínua Educação'
})

banda_larga_escolas = pd.DataFrame({
    'dependencia': ['Federal', 'Estadual', 'Municipal', 'Privada'],
    'percentual': [95.7, 65.1, 73.5, 90.0],
    'ano': 2024,
    'etapa': 'Educação infantil',
    'fonte': 'INEP - Censo Escolar 2024, Resumo Técnico, gráfico 52'
})

print('Tabelas criadas a partir dos valores fornecidos; populações e fontes identificadas.')

## 3. Tratamento e limpeza
Os valores são numéricos e os rótulos foram mantidos próximos ao resumo original. Nesta etapa, uniformizamos tipos e nomes de colunas sem preencher lacunas nem recodificar as categorias. A soma de 99,6% nas quatro condições de estudo/ocupação é compatível com arredondamento a uma casa decimal, mas deve ser confirmada na tabela original.

In [ ]:
internet['ano'] = pd.to_numeric(internet['ano'], errors='raise').astype('int64')
internet['uso_internet_percentual'] = pd.to_numeric(internet['uso_internet_percentual'], errors='raise')
for tabela, coluna in [(internet_recortes, 'percentual'), (estudo_trabalho, 'percentual'), (banda_larga_escolas, 'percentual')]:
    tabela[coluna] = pd.to_numeric(tabela[coluna], errors='raise')
for ano in (2016, 2024):
    escolaridade[ano] = pd.to_numeric(escolaridade[ano], errors='raise')

assert internet['ano'].is_monotonic_increasing
assert internet['uso_internet_percentual'].between(0, 100).all()
assert estudo_trabalho['percentual'].between(0, 100).all()
assert banda_larga_escolas['percentual'].between(0, 100).all()
print('Tipos e faixas conferidos.')
print(f'Soma das categorias estudo/ocupação: {estudo_trabalho.percentual.sum():.1f}% (diferença de arredondamento: {100 - estudo_trabalho.percentual.sum():.1f} p.p.).')

## 4. Valores ausentes e duplicatas
A verificação abaixo procura problemas na transcrição e na organização tabular. Ela não comprova a correção dos valores frente à publicação; essa validação depende da consulta às fontes originais.

In [ ]:
tabelas = {
    'internet': internet, 'internet_recortes': internet_recortes,
    'estudo_trabalho': estudo_trabalho, 'escolaridade': escolaridade,
    'banda_larga_escolas': banda_larga_escolas
}
for nome, tabela in tabelas.items():
    print(f'{nome}: ausentes={int(tabela.isna().sum().sum())}; linhas duplicadas={int(tabela.duplicated().sum())}; linhas={len(tabela)}')

assert all(tabela.isna().sum().sum() == 0 for tabela in tabelas.values())
print('Nenhum valor ausente ou linha duplicada nas tabelas organizadas.')

## 5. Análise exploratória
A série selecionada mostra aumento do uso da internet ao longo dos anos observados. Os recortes de 2025 permitem descrever diferenças de localização e rede de ensino; a distribuição de estudo e trabalho resume situações distintas; a escolaridade permite observar mudanças entre 2016 e 2024; e o Censo Escolar compara a presença de banda larga na educação infantil. Os recortes são descritivos e não são uma mesma população estatística.

In [ ]:
display(internet)
display(internet_recortes)
display(estudo_trabalho)
display(escolaridade)
display(banda_larga_escolas)

## 6. Indicadores derivados e comparações
As diferenças abaixo são calculadas por subtração entre valores publicados. São diferenças descritivas em pontos percentuais ou anos, não estimativas de efeito.

In [ ]:
uso_2016 = internet.loc[internet['ano'].eq(2016), 'uso_internet_percentual'].iloc[0]
uso_2025 = internet.loc[internet['ano'].eq(2025), 'uso_internet_percentual'].iloc[0]
grupo_menor = escolaridade.loc[escolaridade['grupo'].eq('Menor quartil de renda'), 2024].iloc[0]
grupo_maior = escolaridade.loc[escolaridade['grupo'].eq('Maior quartil de renda'), 2024].iloc[0]
rural = internet_recortes.loc[internet_recortes['grupo'].eq('Área rural'), 'percentual'].iloc[0]
urbana = internet_recortes.loc[internet_recortes['grupo'].eq('Área urbana'), 'percentual'].iloc[0]
gap_renda = float(np.subtract(grupo_maior, grupo_menor))
gap_internet = float(np.subtract(urbana, rural))
gap_escolas = float(np.subtract(banda_larga_escolas['percentual'].max(), banda_larga_escolas['percentual'].min()))

indicadores_derivados = pd.DataFrame([
    {'indicador': 'Variação da utilização da internet, 2016-2025', 'valor': uso_2025 - uso_2016, 'unidade': 'pontos percentuais'},
    {'indicador': 'Diferença de anos de estudo entre quartis de renda, 2024', 'valor': gap_renda, 'unidade': 'anos'},
    {'indicador': 'Diferença de uso da internet urbana-rural, 2025', 'valor': gap_internet, 'unidade': 'pontos percentuais'},
    {'indicador': 'Amplitude entre dependências na banda larga infantil, 2024', 'valor': gap_escolas, 'unidade': 'pontos percentuais'}
])
display(indicadores_derivados.round(1))

## 7. Visualizações
As figuras usam uma paleta consistente, títulos, unidades e identificação de população e fonte. Cada gráfico responde a uma pergunta específica. PNGs em alta resolução serão exportados na etapa seguinte.

In [ ]:
def salvar_grafico(nome):
    caminho = PASTA_GRAFICOS / nome
    plt.tight_layout()
    plt.savefig(caminho, dpi=260, bbox_inches='tight', facecolor='white')
    plt.show()

# Evolução do uso da internet, população com 10 anos ou mais
fig, ax = plt.subplots(figsize=(10, 4.8))
sns.lineplot(data=internet, x='ano', y='uso_internet_percentual', marker='o', linewidth=3, color=PALETA['turquesa'], ax=ax)
for row in internet.itertuples():
    ax.annotate(f'{row.uso_internet_percentual:.1f}%', (row.ano, row.uso_internet_percentual), xytext=(0, 11), textcoords='offset points', ha='center', fontsize=9)
ax.set(title='Uso da internet cresce entre os anos selecionados', xlabel='Ano de referência', ylabel='Pessoas de 10 anos ou mais (%)', ylim=(55, 100))
ax.text(0, -0.24, 'Fonte: IBGE, PNAD Contínua TIC. Anos selecionados; população de 10 anos ou mais.', transform=ax.transAxes, fontsize=9, color='#60777b')
sns.despine()
salvar_grafico('01_uso_internet_serie.png')

# Condição de estudo/qualificação e ocupação, jovens de 15 a 29 anos
fig, ax = plt.subplots(figsize=(10, 5.6))
dados_ordenados = estudo_trabalho.sort_values('percentual')
sns.barplot(data=dados_ordenados, x='percentual', y='situacao', color=PALETA['turquesa'], ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=5, fontsize=9)
ax.set(title='Estudo e ocupação entre jovens de 15 a 29 anos · 2024', xlabel='Distribuição percentual (%)', ylabel='')
ax.set_xlim(0, 48)
ax.text(0, -0.22, 'Fonte: IBGE, PNAD Contínua Educação 2024. As categorias somam 99,6% por arredondamento.', transform=ax.transAxes, fontsize=9, color='#60777b')
sns.despine()
salvar_grafico('02_estudo_trabalho_2024.png')

# Comparação de escolaridade entre 2016 e 2024
escolaridade_long = escolaridade.melt(id_vars=['grupo'], value_vars=[2016, 2024], var_name='ano', value_name='anos_estudo')
escolaridade_long['ano'] = escolaridade_long['ano'].astype(str)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=escolaridade_long, x='anos_estudo', y='grupo', hue='ano', palette=['#B5DDD3', PALETA['turquesa']], ax=ax)
ax.set(title='Média de anos de estudo dos jovens de 18 a 29 anos', xlabel='Anos de estudo (média)', ylabel='')
ax.legend(title='Ano', frameon=False, loc='lower right')
ax.text(0, -0.15, 'Fonte: IBGE, PNAD Contínua Educação. Grupos selecionados; categorias podem se sobrepor.', transform=ax.transAxes, fontsize=9, color='#60777b')
sns.despine()
salvar_grafico('03_anos_estudo_grupos.png')

# Conectividade residencial/localização e estudantes por rede, recortes de 2025
fig, ax = plt.subplots(figsize=(9, 4.8))
recortes_ordenados = internet_recortes.sort_values('percentual')
cores = [PALETA['coral'] if 'rural' in grupo.lower() else PALETA['turquesa'] for grupo in recortes_ordenados['grupo']]
ax.barh(recortes_ordenados['grupo'], recortes_ordenados['percentual'], color=cores)
for i, valor in enumerate(recortes_ordenados['percentual']): ax.text(valor + .5, i, f'{valor:.1f}%', va='center', fontsize=9)
ax.set(title='Utilização da internet por recorte · 2025', xlabel='Pessoas que utilizaram internet (%)', ylabel='', xlim=(0, 108))
ax.text(0, -0.2, 'Fonte: IBGE, PNAD Contínua TIC 2025. Populações de referência diferem entre recortes.', transform=ax.transAxes, fontsize=9, color='#60777b')
sns.despine()
salvar_grafico('04_internet_recortes_2025.png')

# Banda larga nas escolas de educação infantil
fig, ax = plt.subplots(figsize=(8, 4.5))
escolas_ordem = banda_larga_escolas.sort_values('percentual')
ax.barh(escolas_ordem['dependencia'], escolas_ordem['percentual'], color=PALETA['menta'])
for i, valor in enumerate(escolas_ordem['percentual']): ax.text(valor + .5, i, f'{valor:.1f}%', va='center', fontsize=9)
ax.set(title='Escolas de educação infantil com banda larga · 2024', xlabel='Percentual de escolas (%)', ylabel='', xlim=(0, 108))
ax.text(0, -0.2, 'Fonte: INEP, Censo Escolar 2024, Resumo Técnico, gráfico 52. Recorte exclusivo da educação infantil.', transform=ax.transAxes, fontsize=9, color='#60777b')
sns.despine()
salvar_grafico('05_banda_larga_infantil_2024.png')

## 8. Interpretação dos resultados
A série selecionada de internet sobe de 66,0% em 2016 para 90,5% em 2025. Isso descreve ampliação do uso na população de 10 anos ou mais, não exclusivamente entre jovens; os intervalos entre anos também não são regulares. Em 2025, o valor informado para área urbana supera o rural em **8,5 pontos percentuais**, mas uso não significa conexão adequada ou equipamento próprio.

Na condição de estudo e ocupação de jovens de 15 a 29 anos, **18,5%** foram classificados como não ocupados e sem estudo ou qualificação, enquanto **39,4%** estavam ocupados e sem estudo ou qualificação. A leitura depende da definição operacional usada pela publicação; os percentuais somam 99,6% por arredondamento.

A média de anos de estudo aumentou no Brasil de **11,1** para **11,9** entre 2016 e 2024. Em 2024, a diferença entre o maior e o menor quartil de renda foi **2,9 anos**. Na educação infantil, a diferença entre a dependência com maior e menor percentual de banda larga foi **30,6 pontos percentuais**. Esses achados descrevem desigualdades; não identificam seus mecanismos nem comprovam causalidade.

**Limitações centrais:** valores transcritos ainda exigem conferência nos documentos originais; recortes têm populações distintas; anos selecionados não constituem série anual completa; médias agregadas ocultam variação dentro dos grupos; presença de internet não mede qualidade, equipamentos ou uso pedagógico.

## 9. Conclusões e propostas de intervenção
Os dados apresentados combinam expansão de acesso com diferenças persistentes por território, renda e dependência administrativa. As propostas a seguir respondem aos desafios observados, mas seus efeitos precisam ser avaliados em programas concretos; esta análise não demonstra causalidade.

### 1. Inclusão digital e recursos de estudo
Ampliar conexão estável, equipamentos adequados e espaços comunitários de aprendizagem, priorizando áreas rurais e unidades escolares com menor cobertura. Acompanhar qualidade, disponibilidade e finalidade de uso, não apenas a existência de conexão. Relação com os recortes de utilização da internet e com a banda larga escolar.

### 2. Apoio socioemocional e mentoria escolar
Oferecer acompanhamento de trajetórias, orientação de estudos, mentoria e apoio psicossocial articulado à escola e à comunidade. A busca ativa deve evitar estigmatização e considerar as barreiras enfrentadas por quem concilia trabalho e estudo ou está fora de ambos. Relação com os indicadores de estudo/ocupação e escolaridade.

### 3. Preparação para o futuro
Conectar formação técnica, competências digitais e orientação profissional a oportunidades locais de trabalho digno. Criar percursos flexíveis para conciliação entre estudo, qualificação e trabalho e acompanhar acesso, conclusão e inserção profissional. Relação com a distribuição de estudo e ocupação e com as diferenças de anos de estudo.

In [ ]:
# 9. Exportação das tabelas tratadas, figuras e metadados compatíveis com o site
internet.to_csv(PASTA_DADOS / 'inclusao_digital.csv', index=False, encoding='utf-8-sig')
internet_recortes.to_csv(PASTA_DADOS / 'inclusao_digital_recortes_2025.csv', index=False, encoding='utf-8-sig')
estudo_trabalho.to_csv(PASTA_DADOS / 'estudo_trabalho_jovens_2024.csv', index=False, encoding='utf-8-sig')
escolaridade.to_csv(PASTA_DADOS / 'anos_de_estudo_2016_2024.csv', index=False, encoding='utf-8-sig')
banda_larga_escolas.to_csv(PASTA_DADOS / 'banda_larga_escolas_infantil_2024.csv', index=False, encoding='utf-8-sig')
indicadores_derivados.to_csv(PASTA_DADOS / 'indicadores_derivados.csv', index=False, encoding='utf-8-sig')

FONTES = [
    {'id': 'ibge_tic', 'institution': 'IBGE', 'title': 'PNAD Contínua: Tecnologia da Informação e Comunicação (TIC)', 'url': 'https://www.ibge.gov.br/estatisticas/sociais/educacao/17270-pnad-continua.html', 'note': 'Portal oficial da pesquisa. Conferir publicação, tabela e notas metodológicas do ano de referência.'},
    {'id': 'ibge_educacao', 'institution': 'IBGE', 'title': 'PNAD Contínua: Educação 2024', 'url': 'https://www.ibge.gov.br/estatisticas/sociais/educacao/17270-pnad-continua.html', 'note': 'Portal oficial da PNAD Contínua. Conferir as tabelas de 2024 antes da apresentação.'},
    {'id': 'inep_censo', 'institution': 'INEP', 'title': 'Censo Escolar da Educação Básica 2024: Resumo Técnico', 'url': 'https://www.gov.br/inep/pt-br/areas-de-atuacao/pesquisas-estatisticas-e-indicadores/censo-escolar/resultados', 'note': 'O indicador de banda larga utilizado neste projeto se refere somente à educação infantil.'},
    {'id': 'inep_microdados', 'institution': 'INEP', 'title': 'Microdados do Censo Escolar', 'url': 'https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/censo-escolar', 'note': 'Fonte para análises adicionais, após consulta à documentação e ao dicionário de dados.'}
]
METADADOS = {
    'project': {
        'title': 'Raio-X da Juventude',
        'subtitle': 'Educação, Tecnologia e Oportunidades para o Futuro',
        'data_status': 'Valores transcritos do material fornecido; confira nos documentos originais. A API oficial estava inacessível durante a preparação.',
        'context': {
            'title': 'IBGE detalha em coletivas públicas a programação da CONFEST/CONFEGE sobre juventude e tecnologia',
            'date': '2026-10-29',
            'url': 'https://agenciadenoticias.ibge.gov.br/',
            'note': 'Contextualização institucional, não fonte estatística. O link específico da notícia requer confirmação.'
        }
    },
    'sources': FONTES,
    'indicators': [
        {'id': 'internet_series', 'title': 'Utilização da internet no Brasil', 'source_id': 'ibge_tic', 'reference_years': [int(row.ano) for row in internet.itertuples()], 'population': 'Pessoas de 10 anos ou mais', 'unit': 'Percentual (%)', 'definition': 'Proporção de pessoas de 10 anos ou mais que utilizaram internet no período de referência da PNAD Contínua TIC.', 'limitation': 'Não representa exclusivamente jovens. Anos selecionados; consultar notas metodológicas para comparabilidade.', 'values': [{'year': int(row.ano), 'value': float(row.uso_internet_percentual)} for row in internet.itertuples()]},
        {'id': 'internet_2025_cuts', 'title': 'Utilização da internet por recorte, 2025', 'source_id': 'ibge_tic', 'reference_years': [2025], 'population': 'Moradores de áreas urbanas e rurais; estudantes das redes pública e privada', 'unit': 'Percentual (%)', 'definition': 'Percentual de pessoas que utilizaram internet em cada recorte informado.', 'limitation': 'Populações de referência distintas; não mede qualidade da conexão ou posse de equipamento.', 'values': [{'group': str(row.grupo), 'value': float(row.percentual)} for row in internet_recortes.itertuples()]},
        {'id': 'study_work_2024', 'title': 'Estudo, qualificação e ocupação entre jovens, 2024', 'source_id': 'ibge_educacao', 'reference_years': [2024], 'population': 'Pessoas de 15 a 29 anos; estimativa informada de 48,0 milhões', 'unit': 'Percentual (%)', 'definition': 'Distribuição percentual por combinação de condição de ocupação e estudo ou qualificação.', 'limitation': 'Categorias totalizam 99,6% por arredondamento e devem ser conferidas no quadro original; não equivalem necessariamente à frequência escolar regular.', 'values': [{'group': str(row.situacao), 'value': float(row.percentual)} for row in estudo_trabalho.itertuples()]},
        {'id': 'study_years_2016_2024', 'title': 'Média de anos de estudo, jovens de 18 a 29 anos', 'source_id': 'ibge_educacao', 'reference_years': [2016, 2024], 'population': 'Pessoas de 18 a 29 anos, por Brasil e grupos selecionados', 'unit': 'Anos de estudo (média)', 'definition': 'Média de anos de estudo concluídos para o grupo informado.', 'limitation': 'Grupos raciais, localização e quartil de renda são dimensões distintas e podem se sobrepor.', 'values': [{'group': str(row[0]), '2016': float(row[1]), '2024': float(row[2])} for row in escolaridade[['grupo', 2016, 2024]].itertuples(index=False, name=None)]},
        {'id': 'school_broadband_2024', 'title': 'Escolas de educação infantil com internet banda larga, 2024', 'source_id': 'inep_censo', 'reference_years': [2024], 'population': 'Escolas de educação infantil, por dependência administrativa', 'unit': 'Percentual de escolas (%)', 'definition': 'Proporção de escolas de educação infantil com internet banda larga, por dependência administrativa.', 'limitation': 'Recorte exclusivo da educação infantil; presença de internet não mede velocidade, estabilidade ou equipamentos.', 'values': [{'group': str(row.dependencia), 'value': float(row.percentual)} for row in banda_larga_escolas.itertuples()]}
    ]
}
(PASTA_DADOS / 'indicadores.json').write_text(json.dumps(METADADOS, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'CSVs, JSON e {len(list(PASTA_GRAFICOS.glob("*.png")))} gráficos exportados em {PASTA_PROJETO}')

## 10. Preparação para o GitHub Pages
Os arquivos gerados ficam organizados em `data/` e `assets/graficos/`. No repositório, a página `index.html`, `style.css` e `script.js` já consomem `data/indicadores.json`; copie para lá o JSON exportado e, se desejar disponibilizar os PNGs para download, a pasta de gráficos.

A pasta do site deve manter esta estrutura na raiz publicada do GitHub Pages:
```text
index.html
style.css
script.js
data/indicadores.json
assets/graficos/*.png
```

O pacote ZIP criado abaixo contém tabelas, metadados e gráficos. Os arquivos HTML/CSS/JS do site devem ser publicados a partir deste repositório, não substituídos pelo ZIP de resultados. No GitHub: abra **Settings → Pages**, escolha **Deploy from a branch**, selecione `main` e `/ (root)`, e salve. Aguarde o endereço indicado pelo GitHub Actions. Para testar no Colab, use o servidor HTTP local; não abra `index.html` por `file://`, pois o navegador bloqueia o carregamento do JSON por `fetch`.

In [ ]:
# Cria um pacote de resultados para anexar ao repositório/site
LEIA_ME = '''RAIO-X DA JUVENTUDE - ARQUIVOS DE RESULTADOS

Conteúdo: tabelas CSV, indicadores.json e gráficos PNG exportados pelo notebook.
Os valores foram transcritos do material fornecido e ainda precisam de conferência
visual nos quadros originais das fontes oficiais antes da apresentação.

Para publicar: copie data/indicadores.json para a pasta data/ do repositório e
os PNGs para assets/graficos/. Preserve index.html, style.css e script.js.

Fontes e populações estão descritas em data/indicadores.json e nas células do notebook.
Acesso à internet não equivale a qualidade de conexão ou posse de equipamento.
'''
(PASTA_PROJETO / 'LEIA_ME_PUBLICACAO.txt').write_text(LEIA_ME, encoding='utf-8')
arquivo_zip = PASTA_PROJETO / 'raio_x_juventude_resultados.zip'
with zipfile.ZipFile(arquivo_zip, 'w', compression=zipfile.ZIP_DEFLATED) as pacote:
    for arquivo in (PASTA_PROJETO / 'LEIA_ME_PUBLICACAO.txt', *PASTA_DADOS.glob('*'), *PASTA_GRAFICOS.glob('*.png')):
        pacote.write(arquivo, arquivo.relative_to(PASTA_PROJETO))
print(f'Pacote pronto: {arquivo_zip}')
print('Estrutura exportada:')
for arquivo in sorted(PASTA_PROJETO.rglob('*')):
    if arquivo.is_file(): print(' -', arquivo.relative_to(PASTA_PROJETO))

# No Google Colab, esta linha inicia o download do ZIP para o computador.
try:
    from google.colab import files
    files.download(str(arquivo_zip))
except ImportError:
    print('Fora do Colab: o pacote está no caminho acima.')